# Lab2.1 — Explorarea unui Overlay PYNQ

În Lab1 am utilizat `BaseOverlay("base.bit")` fără să analizăm în detaliu hardware-ul încărcat în Programmable Logic.

În acest task vom inspecta metadatele Overlay-ului și vom identifica IP-urile hardware accesibile din Processing System.


## Obiective

- încărcarea și inspectarea unui Overlay;
- utilizarea `ip_dict`;
- identificarea tipului, adresei fizice și domeniului de adrese ale unui IP;
- înțelegerea rolului fișierului `.hwh` în descrierea hardware-ului;
- identificarea perifericelor AXI GPIO din Base Overlay.


## 1. Încărcarea Base Overlay


In [ ]:
from pynq.overlays.base import BaseOverlay

base = BaseOverlay("base.bit")
print("Base Overlay loaded.")

## 2. Dicționarul IP-urilor

`ip_dict` este construit din metadatele hardware asociate Overlay-ului și descrie IP-urile accesibile din software.


In [ ]:
base.ip_dict

Afișăm doar informațiile esențiale pentru fiecare IP: nume, tip, adresă fizică și domeniul de adrese.


In [ ]:
print(f"{'IP name':35s} {'Type':35s} {'Address':12s} {'Range':12s}")
print("-" * 98)

for name, info in base.ip_dict.items():
    ip_type = info.get("type", "-")
    phys = info.get("phys_addr")
    size = info.get("addr_range")
    phys_txt = hex(phys) if phys is not None else "-"
    size_txt = hex(size) if size is not None else "-"
    print(f"{name:35s} {ip_type:35s} {phys_txt:12s} {size_txt:12s}")

## 3. Identificarea IP-urilor GPIO

Selectăm IP-urile al căror nume sau tip indică un controler GPIO.


In [ ]:
gpio_ips = {}

for name, info in base.ip_dict.items():
    text = (name + " " + info.get("type", "")).lower()
    if "gpio" in text:
        gpio_ips[name] = info

for name, info in gpio_ips.items():
    print(f"{name:30s}  {info.get('type', '-')}")

Pentru laboratoarele următoare vrem să identificăm controlerele pentru LED-uri, butoane și switch-uri fără să presupunem adrese fixe.


In [ ]:
def find_gpio_name(kind):
    kind = kind.lower()
    aliases = {
        "led": ("led",),
        "button": ("button", "btn"),
        "switch": ("switch", "sw"),
    }
    terms = aliases[kind]
    candidates = []
    for name, info in gpio_ips.items():
        n = name.lower()
        if any(term in n for term in terms):
            if kind != "led" or "rgb" not in n:
                candidates.append(name)
    return candidates

print("LED GPIO candidates    :", find_gpio_name("led"))
print("Button GPIO candidates :", find_gpio_name("button"))
print("Switch GPIO candidates :", find_gpio_name("switch"))

## 4. Ierarhii hardware

Unele Overlay-uri organizează IP-urile în ierarhii. PYNQ expune aceste informații prin `hierarchy_dict`.


In [ ]:
base.hierarchy_dict

## 5. Exercițiu

Identificați din `ip_dict` controlerul AXI GPIO asociat LED-urilor și notați:

- numele IP-ului;
- tipul IP-ului;
- adresa fizică de bază;
- domeniul de adrese.

Nu introduceți adresa manual în cod; extrageți-o din `ip_dict`.


In [ ]:
# Scrieți aici soluția


## 6. Întrebări

1. De ce este util ca software-ul să obțină adresele IP-urilor din metadatele Overlay-ului în loc să folosească adrese hard-codate?
2. Ce reprezintă `phys_addr`?
3. Ce reprezintă `addr_range`?
4. Ce se poate modifica în `ip_dict` dacă încărcăm un alt Overlay?
